In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.app.runtime import load_defaults, load_predictor
from src.modeling.phases.quantization import decode_phase

In [ ]:
RUN_DIR = None
BATCH_SIZE = 1

run_dir = Path(RUN_DIR) if RUN_DIR else max(
    [p for p in (ROOT / "run").glob("*") if (p / "diffusion.yaml").is_file()],
    key=lambda p: p.stat().st_mtime,
)
run_dir = run_dir if run_dir.is_absolute() else ROOT / run_dir

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_phases = load_defaults(run_dir / "vae.yaml")["num_phases"]
predictor = load_predictor(run_dir, device=device)
vae = predictor.vae

shape = (BATCH_SIZE, int(vae.latent_ch), int(vae.latent_size), int(vae.latent_size))
with torch.no_grad():
    latent = predictor.sampler.sample(shape)
    generated = vae.decode(latent).detach().cpu()
phase = decode_phase(generated, num_phases)

print("run dir:", run_dir)
print("generated:", tuple(generated.shape), "range:", (float(generated.min()), float(generated.max())))

In [ ]:
items = [("generated", generated[0, 0], "gray", 0, num_phases - 1), ("phase", phase[0], "gray", 0, num_phases - 1)]

fig, axes = plt.subplots(1, 2, figsize=(8, 4), squeeze=False)
for ax, (title, image, cmap, vmin, vmax) in zip(axes.ravel(), items):
    ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()